# 02 — Training Demo

- Visualise augmentations (before/after)
- Model architecture summary
- Learning rate schedule plot
- Short proof-of-concept training run (3 epochs)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from dermnet.config import load_config
from dermnet.model import build_model
from dermnet.transforms import get_train_transforms, get_val_transforms

cfg = load_config('../configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Augmentation Visualisation

In [ ]:
from pathlib import Path

# Load a sample image
data_dir = Path('../') / cfg.data.data_dir
sample_img_path = next(data_dir.rglob('*.[jJpP][pPnN][gG]'), None)

if sample_img_path:
    img = cv2.cvtColor(cv2.imread(str(sample_img_path)), cv2.COLOR_BGR2RGB)
    train_tf = get_train_transforms(cfg.data.image_size)

    MEAN = np.array([0.485, 0.456, 0.406])
    STD  = np.array([0.229, 0.224, 0.225])

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    # Top row: original (resized)
    orig_resized = cv2.resize(img, (cfg.data.image_size, cfg.data.image_size))
    for col in range(6):
        axes[0, col].imshow(orig_resized)
        axes[0, col].axis('off')
        axes[0, col].set_title('Original', fontsize=8)

    # Bottom row: 6 augmented versions
    for col in range(6):
        aug_tensor = train_tf(image=img)['image']
        aug_np = np.clip(aug_tensor.permute(1,2,0).numpy() * STD + MEAN, 0, 1)
        axes[1, col].imshow(aug_np)
        axes[1, col].axis('off')
        axes[1, col].set_title(f'Augmented #{col+1}', fontsize=8)

    plt.suptitle('Train Augmentation Variations', fontsize=13)
    plt.tight_layout()
    plt.savefig('../outputs/figures/augmentation_demo.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No images found — run scripts/download_data.sh first.')

## 2. Model Architecture Summary

In [ ]:
try:
    from torchinfo import summary
    model = build_model(cfg.model.backbone, cfg.data.num_classes, pretrained=False)
    summary(model, input_size=(1, 3, cfg.data.image_size, cfg.data.image_size),
            col_names=['input_size', 'output_size', 'num_params', 'trainable'])
except ImportError:
    model = build_model(cfg.model.backbone, cfg.data.num_classes, pretrained=False)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total params   : {total/1e6:.2f}M')
    print(f'Trainable params: {trainable/1e6:.2f}M')

## 3. Learning Rate Schedule

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

model = build_model(cfg.model.backbone, cfg.data.num_classes, pretrained=False)
opt = AdamW(model.parameters(), lr=cfg.training.learning_rate)
warmup = LinearLR(opt, start_factor=0.1, end_factor=1.0,
                  total_iters=cfg.scheduler.warmup_epochs)
cosine = CosineAnnealingLR(opt,
    T_max=cfg.training.epochs - cfg.scheduler.warmup_epochs,
    eta_min=cfg.scheduler.min_lr)
scheduler = SequentialLR(opt, [warmup, cosine],
                          milestones=[cfg.scheduler.warmup_epochs])

lrs = []
for _ in range(cfg.training.epochs):
    lrs.append(opt.param_groups[0]['lr'])
    scheduler.step()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, cfg.training.epochs+1), lrs, linewidth=2, color='steelblue')
ax.axvline(cfg.scheduler.warmup_epochs, color='red', linestyle='--',
           label=f'Phase 2 start (epoch {cfg.scheduler.warmup_epochs+1})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
ax.set_title('LR Schedule: Linear Warmup + Cosine Annealing')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/lr_schedule_preview.png', dpi=150)
plt.show()

## 4. Quick Smoke Test (1 batch)

In [ ]:
from pathlib import Path
proc = Path('../data/processed')
if (proc / 'train.csv').exists():
    from dermnet.dataset import build_dataloaders
    train_df = pd.read_csv(proc / 'train.csv')
    val_df   = pd.read_csv(proc / 'val.csv')
    test_df  = pd.read_csv(proc / 'test.csv')

    train_loader, val_loader, _ = build_dataloaders(
        train_df, val_df, test_df,
        get_train_transforms(cfg.data.image_size),
        get_val_transforms(cfg.data.image_size),
        batch_size=8, num_workers=0,
        num_classes=cfg.data.num_classes
    )

    imgs, labels = next(iter(train_loader))
    model = build_model(cfg.model.backbone, cfg.data.num_classes, pretrained=False).to(device)
    with torch.no_grad():
        logits = model(imgs.to(device))
    print(f'Input shape : {imgs.shape}')
    print(f'Output shape: {logits.shape}  (batch x {cfg.data.num_classes} classes)')
    print('Forward pass OK!')
else:
    print('Run: python scripts/prepare_data.py  first, then re-run this cell.')